In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
import time
from scipy.stats import ks_2samp, chi2_contingency


from src.models.predict_model import predictorClass
from src.config import MLFLOW_TRACKING_URI_LOCAL, PIPELINE_SCHEMA_VERSION
from src.features.build_features import (
    drop_features,
    add_new_features,
    make_dummies,
)
import mlflow
import requests


from src.models.train_model import verify_model

from pgmpy.estimators import HillClimbSearch, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork

df_raw = pd.read_csv(
    Path.cwd().parent / "data" / "raw" / "IBM_HR_attr.csv",
    index_col="Unnamed: 0",
)

In [2]:
columns_to_drop = [
    "MonthlyIncome",
    "TotalWorkingYears",
    "YearsInCurrentRole",
    "YearsWithCurrManager",
    "PercentSalaryHike",
    "Over18",
    "EmployeeCount",
    "StandardHours",
    "EmployeeNumber",
    "Attrition_No",
]

all_columns = sorted(list(df_raw.columns))
print(
    "=" * 10,
    "all columns (",
    len(all_columns),
    ") ",
    "=" * 10,
)
print(*all_columns, sep=" ")

usefull_columns = sorted(list(set(all_columns).difference(columns_to_drop)))
real_columns_to_drop = sorted(
    list(set(columns_to_drop).intersection(all_columns))
)

print(
    "=" * 10,
    "usefull_columns (",
    len(usefull_columns),
    ") ",
    "=" * 10,
)
print(*usefull_columns, sep=" ")

print(
    "=" * 10,
    "real_columns_to_drop (",
    len(real_columns_to_drop),
    ") ",
    "=" * 10,
)
print(*real_columns_to_drop, sep=" ")

========== all columns ( 35 )  ==========
Age Attrition BusinessTravel DailyRate Department DistanceFromHome Education EducationField EmployeeCount EmployeeNumber EnvironmentSatisfaction Gender HourlyRate JobInvolvement JobLevel JobRole JobSatisfaction MaritalStatus MonthlyIncome MonthlyRate NumCompaniesWorked Over18 OverTime PercentSalaryHike PerformanceRating RelationshipSatisfaction StandardHours StockOptionLevel TotalWorkingYears TrainingTimesLastYear WorkLifeBalance YearsAtCompany YearsInCurrentRole YearsSinceLastPromotion YearsWithCurrManager
========== usefull_columns ( 26 )  ==========
Age Attrition BusinessTravel DailyRate Department DistanceFromHome Education EducationField EnvironmentSatisfaction Gender HourlyRate JobInvolvement JobLevel JobRole JobSatisfaction MaritalStatus MonthlyRate NumCompaniesWorked OverTime PerformanceRating RelationshipSatisfaction StockOptionLevel TrainingTimesLastYear WorkLifeBalance YearsAtCompany YearsSinceLastPromotion
========== real_columns_to

In [3]:
droping_cols = df_raw.loc[:, real_columns_to_drop]
droping_cols.info()
udf = df_raw.loc[:, usefull_columns]

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 0 to 1469
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   EmployeeCount         1470 non-null   int64 
 1   EmployeeNumber        1470 non-null   int64 
 2   MonthlyIncome         1470 non-null   int64 
 3   Over18                1470 non-null   object
 4   PercentSalaryHike     1470 non-null   int64 
 5   StandardHours         1470 non-null   int64 
 6   TotalWorkingYears     1470 non-null   int64 
 7   YearsInCurrentRole    1470 non-null   int64 
 8   YearsWithCurrManager  1470 non-null   int64 
dtypes: int64(8), object(1)
memory usage: 114.8+ KB


In [4]:
for jj, col in enumerate(droping_cols.columns):
    un = droping_cols[col].unique()
    unl = len(un)
    print(
        f"{jj + 1}) '{col}' has {unl} unique values ({droping_cols[col].dtype}): ",
        end="",
    )
    if unl > 20:
        print("<too many to show>")
    else:
        print(un)

1) 'EmployeeCount' has 1 unique values (int64): [1]
2) 'EmployeeNumber' has 1470 unique values (int64): <too many to show>
3) 'MonthlyIncome' has 1349 unique values (int64): <too many to show>
4) 'Over18' has 1 unique values (object): ['Y']
5) 'PercentSalaryHike' has 15 unique values (int64): [11 23 15 12 13 20 22 21 17 14 16 18 19 24 25]
6) 'StandardHours' has 1 unique values (int64): [80]
7) 'TotalWorkingYears' has 40 unique values (int64): <too many to show>
8) 'YearsInCurrentRole' has 19 unique values (int64): [ 4  7  0  2  5  9  8  3  6 13  1 15 14 16 11 10 12 18 17]
9) 'YearsWithCurrManager' has 18 unique values (int64): [ 5  7  0  2  6  8  3 11 17  1  4 12  9 10 15 13 16 14]


In [5]:
udf.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1470 entries, 0 to 1469
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EnvironmentSatisfaction   1470 non-null   int64 
 9   Gender                    1470 non-null   object
 10  HourlyRate                1470 non-null   int64 
 11  JobInvolvement            1470 non-null   int64 
 12  JobLevel                  1470 non-null   int64 
 13  JobRole                   1470 non-null   object
 14  JobSatisfaction           147

In [6]:
for jj, col in enumerate(udf.columns):
    un = udf[col].unique()
    unl = len(un)
    print(
        f"{jj + 1}) '{col}' has {unl} unique values ({udf[col].dtype}): ",
        end="",
    )
    if unl > 20:
        print("<too many to show>")
    else:
        print(un)

1) 'Age' has 43 unique values (int64): <too many to show>
2) 'Attrition' has 2 unique values (object): ['Yes' 'No']
3) 'BusinessTravel' has 3 unique values (object): ['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
4) 'DailyRate' has 886 unique values (int64): <too many to show>
5) 'Department' has 3 unique values (object): ['Sales' 'Research & Development' 'Human Resources']
6) 'DistanceFromHome' has 29 unique values (int64): <too many to show>
7) 'Education' has 5 unique values (int64): [2 1 4 3 5]
8) 'EducationField' has 6 unique values (object): ['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
9) 'EnvironmentSatisfaction' has 4 unique values (int64): [2 3 4 1]
10) 'Gender' has 2 unique values (object): ['Female' 'Male']
11) 'HourlyRate' has 71 unique values (int64): <too many to show>
12) 'JobInvolvement' has 4 unique values (int64): [3 2 4 1]
13) 'JobLevel' has 5 unique values (int64): [2 1 3 4 5]
14) 'JobRole' has 9 unique values (object): [

In [25]:
ALL_COLUMNS = [
    "Age",
    "Attrition",
    "BusinessTravel",
    "DailyRate",
    "Department",
    "DistanceFromHome",
    "Education",
    "EducationField",
    "EnvironmentSatisfaction",
    "Gender",
    "HourlyRate",
    "JobInvolvement",
    "JobLevel",
    "JobRole",
    "JobSatisfaction",
    "MaritalStatus",
    "MonthlyRate",
    "NumCompaniesWorked",
    "OverTime",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsSinceLastPromotion",
]

TARGET_COL = "Attrition"


def make_bn_training_frame(
    df: pd.DataFrame,
    columns: list[str] = ALL_COLUMNS,
    max_numeric_states: int = 10,
    n_bins: int = 5,
) -> tuple[pd.DataFrame, dict]:
    """
    Преобразует исходный DataFrame в полностью дискретный DataFrame для Bayesian Network.

    Числовые признаки с большим числом уникальных значений бьются на quantile-bins.
    Числовые признаки с малым числом уникальных значений остаются категориями.
    """

    source = df[columns].copy()

    bn_df = pd.DataFrame(index=source.index)

    meta = {
        "columns": columns,
        "high_card_numeric": {},
        "low_card_numeric": [],
        "categorical": [],
    }

    for col in columns:
        s = source[col]

        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)

            if nunique > max_numeric_states:
                q = min(n_bins, nunique)

                codes = pd.qcut(
                    s,
                    q=q,
                    labels=False,
                    duplicates="drop",
                )

                bn_df[col] = codes.astype("Int64").astype(str)

                values_by_state = {}
                for state in sorted(
                    bn_df[col].dropna().unique(), key=lambda x: int(x)
                ):
                    mask = bn_df[col] == state
                    values_by_state[state] = (
                        s[mask].dropna().astype(int).to_numpy()
                    )

                meta["high_card_numeric"][col] = {
                    "values_by_state": values_by_state,
                    "fallback_values": s.dropna().astype(int).to_numpy(),
                }

            else:
                bn_df[col] = s.astype("Int64").astype(str)
                meta["low_card_numeric"].append(col)

        else:
            bn_df[col] = s.astype(str)
            meta["categorical"].append(col)

    return bn_df, meta


def fit_bayesian_generator(
    df: pd.DataFrame,
    max_indegree: int = 3,
    max_iter: int = 2_000,
    n_bins: int = 5,
    equivalent_sample_size: int = 5,
    scoring_method="bic-d",
):
    """
    1. Дискретизирует данные.
    2. Учит структуру Bayesian Network.
    3. Оценивает CPD.
    """

    bn_df, meta = make_bn_training_frame(
        df=df,
        n_bins=n_bins,
    )

    start = time.perf_counter()

    structure_estimator = HillClimbSearch(data=bn_df)

    dag = structure_estimator.estimate(
        scoring_method="bic-d",
        max_indegree=max_indegree,
        max_iter=max_iter,
        show_progress=True,
    )

    model = DiscreteBayesianNetwork(ebunch=list(dag.edges()))
    model.add_nodes_from(bn_df.columns)

    estimator = BayesianEstimator(model, bn_df)

    cpds = estimator.get_parameters(
        prior_type="BDeu",
        equivalent_sample_size=equivalent_sample_size,
        n_jobs=1,
    )

    model.add_cpds(*cpds)
    model.check_model()

    fit_seconds = time.perf_counter() - start

    return model, meta, fit_seconds


def decode_bn_samples(
    samples_bn: pd.DataFrame,
    meta: dict,
    seed: int | None = None,
) -> pd.DataFrame:
    """
    Возвращает сгенерированные BN данные обратно в формат, похожий на исходный:
    bins -> реальные int значения из соответствующих bin-ов.
    """

    rng = np.random.default_rng(seed)
    result = pd.DataFrame(index=samples_bn.index)

    for col in meta["columns"]:
        if col in meta["high_card_numeric"]:
            info = meta["high_card_numeric"][col]
            values = []

            for state in samples_bn[col].astype(str):
                pool = info["values_by_state"].get(state)

                if pool is None or len(pool) == 0:
                    pool = info["fallback_values"]

                values.append(int(rng.choice(pool)))

            result[col] = values

        elif col in meta["low_card_numeric"]:
            result[col] = pd.to_numeric(
                samples_bn[col], errors="coerce"
            ).astype(int)

        else:
            result[col] = samples_bn[col].astype(str)

    return result


def generate_synthetic_data(
    model: DiscreteBayesianNetwork,
    meta: dict,
    n_samples: int = 100,
    seed: int | None = 42,
) -> pd.DataFrame:
    """
    Генерирует синтетические данные в исходном формате.
    """

    samples_bn = model.simulate(
        n_samples=n_samples,
        seed=seed,
        show_progress=False,
    )

    return decode_bn_samples(samples_bn, meta, seed=seed)


def make_data_drift(
    df: pd.DataFrame,
    strength: float = 0.5,
    seed: int | None = 42,
) -> pd.DataFrame:
    """
    Искусственный data drift для демонстрации.

    Меняем распределения входных признаков:
    - больше OverTime = Yes
    - больше Travel_Frequently
    - ниже EnvironmentSatisfaction
    - выше DistanceFromHome
    - больше Single

    Target Attrition лучше не трогать: это уже не data drift, а target/concept drift.
    """

    rng = np.random.default_rng(seed)
    drifted = df.copy()
    n = len(drifted)

    mask = rng.random(n) < strength

    if "OverTime" in drifted.columns:
        drifted.loc[mask, "OverTime"] = rng.choice(
            ["Yes", "No"],
            size=mask.sum(),
            p=[0.75, 0.25],
        )

    if "BusinessTravel" in drifted.columns:
        drifted.loc[mask, "BusinessTravel"] = rng.choice(
            ["Travel_Frequently", "Travel_Rarely", "Non-Travel"],
            size=mask.sum(),
            p=[0.65, 0.30, 0.05],
        )

    if "EnvironmentSatisfaction" in drifted.columns:
        drifted.loc[mask, "EnvironmentSatisfaction"] = rng.choice(
            [1, 2, 3, 4],
            size=mask.sum(),
            p=[0.45, 0.35, 0.15, 0.05],
        )

    if "DistanceFromHome" in drifted.columns:
        drifted.loc[mask, "DistanceFromHome"] = rng.choice(
            np.arange(15, 30),
            size=mask.sum(),
        )

    if "MaritalStatus" in drifted.columns:
        drifted.loc[mask, "MaritalStatus"] = rng.choice(
            ["Single", "Married", "Divorced"],
            size=mask.sum(),
            p=[0.65, 0.25, 0.10],
        )

    return drifted


def make_predict_payloads(df: pd.DataFrame) -> list[dict]:
    """
    Подготовка строк для POST /predict.
    Attrition убираем, потому что это target.
    """

    features = df.drop(columns=[TARGET_COL], errors="ignore")
    return features.to_dict(orient="records")

In [28]:
bn_model, bn_meta, fit_seconds = fit_bayesian_generator(
    udf, max_indegree=50, n_bins=50, max_iter=2_000, scoring_method="bdeu"
)

print(f"Bayesian Network fitted in {fit_seconds:.2f} seconds")
print("Edges:", list(bn_model.edges())[:20])

C:\Users\Mishas\AppData\Local\Temp\ipykernel_15980\3582223481.py:117: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  structure_estimator = HillClimbSearch(data=bn_df)
  1%|          | 11/2000 [00:03<09:29,  3.49it/s] 
C:\Users\Mishas\AppData\Local\Temp\ipykernel_15980\3582223481.py:129: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  estimator = BayesianEstimator(model, bn_df)


Bayesian Network fitted in 3.41 seconds
Edges: [('Attrition', 'OverTime'), ('Attrition', 'MaritalStatus'), ('Attrition', 'BusinessTravel'), ('Attrition', 'JobInvolvement'), ('MaritalStatus', 'StockOptionLevel'), ('Department', 'EducationField'), ('JobLevel', 'JobRole'), ('JobLevel', 'Attrition'), ('JobRole', 'Department'), ('YearsAtCompany', 'JobLevel'), ('YearsSinceLastPromotion', 'YearsAtCompany')]


In [13]:
df_syn = generate_synthetic_data(
    model=bn_model,
    meta=bn_meta,
    n_samples=1470,  # same as df len part
    seed=123,
)

for col in droping_cols.columns:
    df_syn[col] = droping_cols[col]

In [14]:
df_syn.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,YearsSinceLastPromotion,EmployeeCount,EmployeeNumber,MonthlyIncome,Over18,PercentSalaryHike,StandardHours,TotalWorkingYears,YearsInCurrentRole,YearsWithCurrManager
0,37,No,Travel_Rarely,1327,Research & Development,2,2,Medical,1,Female,...,2,1,1,5993,Y,11,80,8,4,5
1,29,No,Travel_Rarely,1323,Sales,24,4,Technical Degree,2,Male,...,0,1,2,5130,Y,23,80,10,7,7
2,40,No,Travel_Rarely,482,Research & Development,5,2,Other,3,Male,...,0,1,4,2090,Y,15,80,7,0,0
3,45,No,Travel_Rarely,1448,Research & Development,3,2,Life Sciences,1,Male,...,0,1,5,2909,Y,11,80,8,7,0
4,43,No,Travel_Rarely,481,Sales,9,3,Life Sciences,4,Male,...,2,1,7,3468,Y,12,80,6,2,2


In [15]:
df_raw.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [16]:
try:
    requests.get(MLFLOW_TRACKING_URI_LOCAL, timeout=10.0)
except requests.RequestException:
    print("MLflow server unavailible, ", end="")
    print("predictorClass tests are skipped")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_LOCAL)
runs = mlflow.search_runs(
    search_all_experiments=True,
    filter_string=(
        "tags.pipeline_schema_version = "
        f"'{PIPELINE_SCHEMA_VERSION}' AND tags.model_saved = 'True'"
    ),
)

if len(runs) == 0:
    print("There are no suitable models in mlflow, ", end="")
    print("predictorClass tests are skipped")
else:
    run_id = runs.iloc[0]["run_id"]
    model_uri = f"runs:/{run_id}/model"
    model = mlflow.sklearn.load_model(model_uri)

In [17]:
def idiotic_func(x):
    return 0 if x == "No" else 1


df_raw_y = df_raw["Attrition"].copy().apply(idiotic_func)
df_syn_y = df_syn["Attrition"].copy().apply(idiotic_func)

In [18]:
def preproc(df):
    df = add_new_features(df)
    df = make_dummies(df)
    df = drop_features(df)
    return df


raw_x = preproc(df_raw)
syn_x = preproc(df_syn)

In [19]:
raw_results = verify_model(model, raw_x, df_raw_y)
print("===" * 10)
print(raw_results)

Оптимальный порог по f1-score: 1.0000
F1 Score: 0.5406976744186046
ROC AUC: 0.7849435872165245
PR AUC: 0.35836279893528605
Confusion matrix: 
 [[968 265]
 [ 51 186]]
{'f1_Score': 0.5406976744186046, 'precision': 0.4124168514412417, 'recall': 0.7848101265822784, 'roc_auc': 0.7849435872165245, 'pr_auc': 0.35836279893528605}


In [20]:
syn_results = verify_model(model, syn_x, df_syn_y)
print("===" * 10)
print(syn_results)

Оптимальный порог по f1-score: 1.0000
F1 Score: 0.33127317676143386
ROC AUC: 0.584081766375484
PR AUC: 0.21279862632388433
Confusion matrix: 
 [[795 412]
 [129 134]]
{'f1_Score': 0.33127317676143386, 'precision': 0.2454212454212454, 'recall': 0.5095057034220533, 'roc_auc': 0.584081766375484, 'pr_auc': 0.21279862632388433}


In [31]:
FEATURE_COLUMNS = [
    "Age",
    "BusinessTravel",
    "DailyRate",
    "Department",
    "DistanceFromHome",
    "Education",
    "EducationField",
    "EnvironmentSatisfaction",
    "Gender",
    "HourlyRate",
    "JobInvolvement",
    "JobLevel",
    "JobRole",
    "JobSatisfaction",
    "MaritalStatus",
    "MonthlyRate",
    "NumCompaniesWorked",
    "OverTime",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "TrainingTimesLastYear",
    "WorkLifeBalance",
    "YearsAtCompany",
    "YearsSinceLastPromotion",
]


CATEGORICAL_COLUMNS = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime",
]


def compare_feature_drift(
    real_df: pd.DataFrame,
    synthetic_df: pd.DataFrame,
    feature_columns: list[str] = FEATURE_COLUMNS,
    categorical_columns: list[str] = CATEGORICAL_COLUMNS,
    alpha: float = 0.05,
    min_expected_count: int = 5,
) -> pd.DataFrame:
    """
    Сравнивает распределения признаков в real_df и synthetic_df.

    Для числовых признаков:
        KS-test.

    Для категориальных признаков:
        Chi-square test.

    Возвращает DataFrame с p-value и флагом drift_detected.
    """

    results = []

    categorical_columns = set(categorical_columns)

    for col in feature_columns:
        if col not in real_df.columns or col not in synthetic_df.columns:
            results.append(
                {
                    "feature": col,
                    "test": "missing_column",
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "drift_detected": None,
                    "note": "column missing in one of dataframes",
                }
            )
            continue

        real_s = real_df[col].dropna()
        synth_s = synthetic_df[col].dropna()

        if len(real_s) == 0 or len(synth_s) == 0:
            results.append(
                {
                    "feature": col,
                    "test": "empty_column",
                    "statistic": np.nan,
                    "p_value": np.nan,
                    "drift_detected": None,
                    "note": "empty after dropna",
                }
            )
            continue

        if col in categorical_columns:
            all_categories = sorted(
                set(real_s.astype(str).unique())
                | set(synth_s.astype(str).unique())
            )

            real_counts = (
                real_s.astype(str)
                .value_counts()
                .reindex(all_categories, fill_value=0)
            )
            synth_counts = (
                synth_s.astype(str)
                .value_counts()
                .reindex(all_categories, fill_value=0)
            )

            contingency = np.vstack([real_counts.values, synth_counts.values])

            try:
                stat, p_value, _, expected = chi2_contingency(contingency)

                note = ""
                if (expected < min_expected_count).any():
                    note = "some expected counts are low; chi-square may be unstable"

                results.append(
                    {
                        "feature": col,
                        "test": "chi2",
                        "statistic": stat,
                        "p_value": p_value,
                        "drift_detected": bool(p_value < alpha),
                        "note": note,
                    }
                )

            except ValueError as e:
                results.append(
                    {
                        "feature": col,
                        "test": "chi2",
                        "statistic": np.nan,
                        "p_value": np.nan,
                        "drift_detected": None,
                        "note": str(e),
                    }
                )

        else:
            real_num = pd.to_numeric(real_s, errors="coerce").dropna()
            synth_num = pd.to_numeric(synth_s, errors="coerce").dropna()

            if len(real_num) == 0 or len(synth_num) == 0:
                results.append(
                    {
                        "feature": col,
                        "test": "ks",
                        "statistic": np.nan,
                        "p_value": np.nan,
                        "drift_detected": None,
                        "note": "non-numeric values after conversion",
                    }
                )
                continue

            stat, p_value = ks_2samp(real_num, synth_num)

            results.append(
                {
                    "feature": col,
                    "test": "ks",
                    "statistic": stat,
                    "p_value": p_value,
                    "drift_detected": bool(p_value < alpha),
                    "note": "",
                }
            )

    result_df = pd.DataFrame(results)

    result_df = result_df.sort_values(
        by=["drift_detected", "p_value"],
        ascending=[False, True],
        na_position="last",
    ).reset_index(drop=True)

    return result_df

In [34]:
drift_report = compare_feature_drift(
    real_df=df_raw,
    synthetic_df=df_syn,
    alpha=0.05,
)

drift_report

e:\miniconda\envs\mlops\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,feature,test,statistic,p_value,drift_detected,note
0,OverTime,chi2,1.782578,0.181833,False,
1,Department,chi2,1.983745,0.370882,False,
2,EducationField,chi2,4.453396,0.486143,False,
3,MonthlyRate,ks,0.027891,0.616987,False,
4,MaritalStatus,chi2,0.566663,0.753270,False,
5,JobSatisfaction,ks,0.024490,0.770298,False,
6,JobLevel,ks,0.023810,0.799134,False,
7,YearsAtCompany,ks,0.023810,0.799134,False,
8,BusinessTravel,chi2,0.389797,0.822918,False,
9,HourlyRate,ks,0.022449,0.852851,False,


In [35]:
df_syn_drifted = make_data_drift(df=df_syn)

In [36]:
drift_report = compare_feature_drift(
    real_df=df_raw,
    synthetic_df=df_syn_drifted,
    alpha=0.05,
)

drift_report

e:\miniconda\envs\mlops\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


,feature,test,statistic,p_value,drift_detected,note
0,DistanceFromHome,ks,0.387755,6.103324e-99,True,
1,OverTime,chi2,185.730368,2.718654e-42,True,
2,BusinessTravel,chi2,187.912459,1.567934e-41,True,
3,EnvironmentSatisfaction,ks,0.220408,1.100978e-31,True,
4,MaritalStatus,chi2,96.217677,1.278194e-21,True,
5,Department,chi2,1.983745,3.708815e-01,False,
6,EducationField,chi2,4.453396,4.861431e-01,False,
7,MonthlyRate,ks,0.027891,6.169868e-01,False,
8,JobSatisfaction,ks,0.024490,7.702983e-01,False,
9,JobLevel,ks,0.023810,7.991341e-01,False,
